In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
from matplotlib import pyplot
import matplotlib.pylab as plt
import time
from datetime import timedelta
start = time.perf_counter()
from sklearn.metrics import mean_absolute_error as mae
from sklearn.metrics import mean_squared_error as mse
from sklearn.metrics import mean_absolute_percentage_error as mape
#Essas bibliotecas implementam o modelo ANFIS e devem estar na mesma pasta.
!pip install membership
!pip install anfis
!pip install funciones_entrenamiento2
import importlib
import funciones_entrenamiento2
importlib.reload(funciones_entrenamiento2)
import membership
import anfis
import funciones_entrenamiento2

## Exemplo

In [ ]:
# 1. carrega os dados de entrada e depois o de saída (que o código vai buscar)
df2 = pd.read_excel("Dem_anual_1953.xlsx")
df = pd.read_excel("Dem_anual__t-1.xlsx")

# 2.  Converte para array NumPy (se necessário)
array_dados = df.values
array_dados2 = df2.values
# 3. Aplicar reshape (ex: reshape para uma matriz 2D)

y = array_dados2.reshape(len(array_dados), -1)  # -1 infere o tamanho da outra dimensão
X = array_dados.reshape(len(array_dados), -1)
# 4. (Opicional) Manipular o novo array
# ...
# exibir o novo array (opcional)
#print('x')
#print(X)

# Geração dos conjuntos de treinamento,validação e teste
prop_tr = 0.7
prop_val = 0.2

ind_tr = int(y.shape[0] * prop_tr)
ind_val = ind_tr + int(y.shape[0] * prop_val)

# Esses print's são apenas para conferir se está correto
#print(ind_tr)
#print(ind_val)
#print(' ')
X_tr = X[:ind_tr]
X_val = X[ind_tr:ind_val]
X_test = X[ind_val:]
y_tr = y[:ind_tr]
y_val = y[ind_tr:ind_val]
y_test =y[ind_val:]

#print(' ')
#print(y_tr)
#print(' ')
#print(y_val)
#print(y_test)
#print(' ')
#print(X_tr.shape)
#print(X_val.shape)
#print(X_test.shape)
#print(' ')
#print(y_tr.shape)
#print(y_val.shape)
#print(y_test.shape)

# Normalização dos dados, por máximo e mínimo (dependo o que fica melhor para seu caso)
#scaler = MinMaxScaler()
#X_tr = scaler.fit_transform(X_tr)
#X_val = scaler.transform(X_val)
#X_test = scaler.transform(X_test)

#y_tr = scaler.fit_transform(y_tr)
#y_val = scaler.transform(y_val)
#y_test = scaler.transform(y_test)

## Normalização dos dados, por Escala Logaritmica (dependo o que fica melhor para seu caso)
X_tr = np.log(X_tr)
X_val = np.log(X_val)
X_test = np.log(X_test)

y_tr = np.log(y_tr)
y_val = np.log(y_val)
y_test = np.log(y_test)

# Definir variaveis em formato pytorch
X_tr_tensor = torch.tensor(X_tr).float()
X_val_tensor = torch.tensor(X_val).float()
X_test_tensor = torch.tensor(X_test).float()

y_tr_tensor = torch.tensor(y_tr).float()
y_val_tensor = torch.tensor(y_val).float()
y_test_tensor = torch.tensor(y_test).float()

#print(X_tr_tensor.shape)
#print(X_val_tensor.shape)
#print(X_test_tensor.shape)
#print (' ')
#print(y_tr_tensor.shape)
#print(y_val_tensor.shape)
#print(y_test_tensor.shape)
#print(' ')
#print(X_tr_tensor)
#print(X_val_tensor)
#print(X_test_tensor)
#print (' ')
#print(y_tr_tensor)
#print(y_val_tensor)
#print(' ')
#print(y_test_tensor)

force_cpu = True

if(torch.cuda.is_available() and not(force_cpu)):
    device = 'cuda'
else:
    device = 'cpu'
print(device)

# Parâmetros
num_mfs = 4
num_out = 1
hybrid = False
num_rules = 10000 #limite
initial_rules = None
feature_importance = None
random_state = 0 #1,2,3,...
null_values = True

# Definir modelo
import importlib
importlib.reload(anfis)
importlib.reload(funciones_entrenamiento2)
importlib.reload(membership)
model = membership.make_anfis(X_tr_tensor, num_mfs = num_mfs, num_out = num_out, hybrid = hybrid,
                                      num_rules = num_rules, initial_rules = initial_rules,
                                      feature_importance = feature_importance, random_state = random_state,
                                      device = device, null_values = null_values)

# Visualização das regras difusas
rules = model.get_rules()
print(rules.shape)
print(rules[:5])

## Definir a função de agregação
#- 0  Producto
#- 1 Mínimo
#- 2 Integral de sugeno generalizada
#- 3 OWA
#- 4 F1, alpha = 0.1
#- 5 F1, alpha = 0.25
#- 6 F1, alpha = 0.5
#- 7 F2, k = 1.1
#- 8 F2, k = 1.2
#- 9 F3, alpha = 0.5
#- 10 F3, alpha = 0.7
#- 11 F3, alpha = 0.9
#- 12 Media geométrica
#- 13 Overlap 1
#- 14 Overlap 2

model.layer['rules'].option = 0

# Definir data loaders
BATCH_SIZE = 64

td = TensorDataset(X_tr_tensor, y_tr_tensor)
td_val = TensorDataset(X_val_tensor, y_val_tensor)
td_test = TensorDataset(X_test_tensor, y_test_tensor)

data_train = DataLoader(td, batch_size = BATCH_SIZE, shuffle = False)
data_val = DataLoader(td_val, batch_size = BATCH_SIZE, shuffle = False)
data_test = DataLoader(td_test, batch_size = BATCH_SIZE, shuffle = False)

# Definir função perda
loss_function = torch.nn.MSELoss() # MSE
loss_mae = torch.nn.L1Loss() #MAE

n_epochs = 100000

optimizer = torch.optim.Adam([
                    {"params": model.layer['fuzzify'].parameters(), "lr": 0.0005},
                    {"params": model.layer['rules'].truth_rules, "lr": 0.005},
                    {"params": model.layer['rules'].sugeno_weights, "lr": 0.001},
                    {"params": model.layer['consequent'].parameters()},],
                lr = 0.5)

# Defini a parada antecipada, caso fique 30.000 interações sem o MSE alterar até uma tolerância 1e-06
funciones_entrenamiento2.train_anfis(model, data_train, data_val, optimizer,
                                       loss_function, loss_mae, n_epochs,
                                       ruta_modelo = './ANFIS_GTSK.pth',
                                       paciencia = 30000, limiar_tolerancia = 1e-6)

# Carrega o melhor modelo de validação
model = torch.load('/content/ANFIS_GTSK.pth', weights_only=False)

pred_test = funciones_entrenamiento2.batch_predict(model, data_test)

# Apresenta as métricas de erro para a análise
test_loss = loss_function(y_test_tensor, pred_test).item()
print('\nError test (MSE):', round(test_loss, 7))

rmse = np.sqrt(test_loss)
print('\nError test (RMSE):', round(rmse, 7))

mae = mae(y_test_tensor, pred_test)
print('\nError test (MAE):', round(mae, 7))

mape = mape(y_test_tensor, pred_test)*100
print('\nError test (MAPE):', round(mape, 7))

print(' ')
print('Previsão:\n', np.round(pred_test.detach().cpu().numpy(), 8))
print(' ')

qt = len(pred_test)
print('Quantidade de valores previstos: ', qt)
print(' ')

#Era para plotar os gráficos da previsão, mas acabei não utilizando, pode ter que ajustar 
# ou ter algo mau escrito no código, pois acabei fazendo ajustes no caminho e não alterei aqui.

#plt.plot(pred_test, color = 'red', label = 'ANFIS Teste')
#plt.plot(y_test_tensor, color = 'blue', label = 'Saída Teste')
#plt.title('Predição - Modelo ANFIS x Saída')
#plt.xlabel('Ano')
#plt.ylabel('Valores em escala Logarítmica')
#plt.legend()
#plt.grid(True)
#plt.savefig('anfis_model.jpg')
#plt.show()

#plt.plot(y, color = 'red', label = 'Real')
#plt.plot(model, color = 'blue', label = 'Model')
#plt.title(' Real')
#plt.xlabel('Ano')
#plt.ylabel('Valor')
#plt.legend()
#plt.grid(True)
#plt.savefig('model.jpg')
#plt.show()

#print(rules.shape)
#print(' ')
#print(rules[:5])
#print(' ')
#print(rules)

# Tempo de execução
end = time.perf_counter()
print(f"\nTempo total de execução: {end - start:.2f} segundos")


# =================================================================================
# ACESSO DETALHADO ÀS REGRAS DO CONJUNTO FUZZY (ainda tem coisa para melhorar aqui)
# =================================================================================

# 1. Extração dos dados e tratamento seguro dos tensores do PyTorch
matriz_antecedentes = rules.cpu().numpy()
pesos_ativacao = model.layer['rules'].truth_rules.detach().cpu().numpy().flatten()

# Extrai os parâmetros da camada consequent e garante que sejam arrays bidimensionais normais
params_lista = list(model.layer['consequent'].parameters())
pesos_consequent = params_lista[0].detach().cpu().numpy()

# Se a camada separar o bias em outro parâmetro, é capturado aqui
bias_consequent = None
if len(params_lista) > 1:
    bias_consequent = params_lista[1].detach().cpu().numpy().flatten()

num_entradas = matriz_antecedentes.shape[1]
nome_arquivo = "regras_fuzzy.txt"

print(f"\nEscrevendo a estrutura real de todas as {len(matriz_antecedentes)} regras no arquivo '{nome_arquivo}'...")

with open(nome_arquivo, "w", encoding="utf-8") as f:
    f.write("=====================================================================\n")
    f.write(f"           ESTRUTURA COMPLETA DAS REGRAS FUZZY (ANFIS)               \n")
    f.write("=====================================================================\n\n")

    for i, linha_antecedente in enumerate(matriz_antecedentes):
        # Construindo o ANTECEDENTE (IF)
        partes_if = []
        for var_idx, mf_idx in enumerate(linha_antecedente):
            partes_if.append(f"Var_Entrada_{var_idx} é MF_{mf_idx}")
        texto_if = " SE " + " E ".join(partes_if)

        # Construindo o CONSEQUENTE (THEN) tratando os valores estritamente como floats simples
        partes_then = []

        # Isola a linha de coeficientes da regra atual
        if pesos_consequent.ndim > 1:
            coeficientes_regra = pesos_consequent[i].flatten()
        else:
            coeficientes_regra = pesos_consequent.flatten()

        for var_idx in range(num_entradas):
            # Força a conversão para float nativo do Python para evitar erros de formatação de string do numpy
            coef = float(coeficientes_regra[var_idx])
            partes_then.append(f"({coef:.4f} * Var_Entrada_{var_idx})")

        # Define o termo independente (bias) da regra
        if bias_consequent is not None:
            bias_valor = float(bias_consequent[i]) if i < len(bias_consequent) else float(bias_consequent[0])
        else:
            # Caso o bias esteja embutido como a última coluna da matriz principal
            bias_valor = float(coeficientes_regra[-1])

        partes_then.append(f"({bias_valor:.4f})")

        texto_then = " ENTÃO Saída = " + " + ".join(partes_then)

        # Grau de ativação da regra
        peso_regra = float(pesos_ativacao[i]) if i < len(pesos_ativacao) else 1.0
        texto_peso = f" [Peso/Grau de Ativação: {peso_regra:.6f}]"

        # Monta a string final e escreve
        linha_completa = f"REGRA {i:03d}:{texto_if}{texto_then}{texto_peso}\n"
        f.write(linha_completa)

        # Exibe as 10 primeiras linhas de amostra no console
        if i < 10:
            print(linha_completa.strip())

print(f"\n[Sucesso] O arquivo '{nome_arquivo}' foi gerado!")
# =====================================================================
with open("previsao.txt", "w") as arquivo:
    # Transforma o Tensor em uma lista comum do Python e depois em texto
    arquivo.write(str(pred_test.tolist()))
